# SGB — Sistema de Gestión de Biblioteca
Proyecto POO · Python 3.10+ · Google Colab · Arquitectura de 3 capas (Modelo, Servicios, Repositorios)

## Celda 0 — Configuración

In [ ]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

# Ruta base del proyecto en Drive
RUTA_BASE = '/content/drive/MyDrive/SGB/datos/'
os.makedirs(RUTA_BASE, exist_ok=True)

# Rutas de cada archivo de persistencia
RUTA_USUARIOS   = RUTA_BASE + 'usuarios.txt'
RUTA_LIBROS     = RUTA_BASE + 'libros.txt'
RUTA_CATEGORIAS = RUTA_BASE + 'categorias.txt'
RUTA_PRESTAMOS  = RUTA_BASE + 'prestamos.txt'
RUTA_MULTAS     = RUTA_BASE + 'multas.txt'
RUTA_RESERVAS   = RUTA_BASE + 'reservas.txt'

# Constantes de negocio
SEPARADOR = '|'
MULTA_POR_DIA = 1000  # COP por cada día de retraso

## Sección 1 — Modelo (dominio)

### Enums

In [ ]:
from enum import Enum


class TipoUsuario(Enum):
    """Tipo de usuario con su plazo de préstamo (días) y cupo de libros."""
    ESTUDIANTE = ("estudiante", 7, 3)
    DOCENTE = ("docente", 15, 5)
    EXTERNO = ("externo", 5, 2)

    def __init__(self, etiqueta, dias_prestamo, cupo):
        self.etiqueta = etiqueta
        self.dias_prestamo = dias_prestamo
        self.cupo = cupo

    @classmethod
    def desde_texto(cls, texto):
        """Convierte un texto ('estudiante', 'docente', 'externo') en TipoUsuario."""
        for t in cls:
            if t.etiqueta == texto.strip().lower():
                return t
        raise ValueError(f"Tipo de usuario inválido: {texto}")


class EstadoUsuario(Enum):
    """Estado de un usuario en el sistema."""
    ACTIVO = "activo"
    SUSPENDIDO = "suspendido"


class EstadoReserva(Enum):
    """Estado de una reserva de libro."""
    ACTIVA = "activa"
    CANCELADA = "cancelada"
    ATENDIDA = "atendida"

### Usuario y Categoria

In [ ]:
class Usuario:
    """Usuario de la biblioteca (estudiante, docente o externo)."""

    def __init__(self, documento, nombre, tipo, correo, telefono,
                 estado=EstadoUsuario.ACTIVO):
        self.documento = documento
        self.nombre = nombre
        self.tipo = tipo
        self.correo = correo
        self.telefono = telefono
        self.estado = estado

    def esta_activo(self):
        """Retorna True si el usuario está activo."""
        return self.estado == EstadoUsuario.ACTIVO

    def to_linea(self):
        """Serializa el usuario a una línea delimitada por pipe."""
        return SEPARADOR.join([self.documento, self.nombre, self.tipo.etiqueta,
                               self.correo, self.telefono, self.estado.value])

    @classmethod
    def desde_linea(cls, linea):
        """Construye un Usuario desde una línea del archivo."""
        d, n, t, c, tel, e = linea.strip().split(SEPARADOR)
        return cls(d, n, TipoUsuario.desde_texto(t), c, tel, EstadoUsuario(e))

    def __str__(self):
        return (f"{self.documento} | {self.nombre} | {self.tipo.etiqueta} | "
                f"{self.correo} | {self.telefono} | {self.estado.value}")


class Categoria:
    """Categoría temática de libros."""

    def __init__(self, id_categoria, nombre):
        self.id_categoria = id_categoria
        self.nombre = nombre

    def to_linea(self):
        """Serializa la categoría a una línea delimitada por pipe."""
        return SEPARADOR.join([str(self.id_categoria), self.nombre])

    @classmethod
    def desde_linea(cls, linea):
        """Construye una Categoria desde una línea del archivo."""
        i, n = linea.strip().split(SEPARADOR)
        return cls(int(i), n)

    def __str__(self):
        return f"[{self.id_categoria}] {self.nombre}"

### Libro

In [ ]:
class Libro:
    """Libro del catálogo con control de ejemplares totales y disponibles."""

    def __init__(self, isbn, titulo, autor, id_categoria, anio,
                 ejemplares_totales, ejemplares_disponibles=None):
        self.isbn = isbn
        self.titulo = titulo
        self.autor = autor
        self.id_categoria = id_categoria
        self.anio = anio
        self.ejemplares_totales = ejemplares_totales
        self.ejemplares_disponibles = (ejemplares_totales
                                       if ejemplares_disponibles is None
                                       else ejemplares_disponibles)

    def hay_disponibles(self):
        """Retorna True si hay al menos un ejemplar disponible."""
        return self.ejemplares_disponibles > 0

    def prestar_ejemplar(self):
        """Descuenta un ejemplar disponible."""
        if not self.hay_disponibles():
            raise ValueError("No hay ejemplares disponibles")
        self.ejemplares_disponibles -= 1

    def devolver_ejemplar(self):
        """Suma un ejemplar disponible (sin superar el total)."""
        if self.ejemplares_disponibles < self.ejemplares_totales:
            self.ejemplares_disponibles += 1

    def to_linea(self):
        """Serializa el libro a una línea delimitada por pipe."""
        return SEPARADOR.join([self.isbn, self.titulo, self.autor,
                               str(self.id_categoria), str(self.anio),
                               str(self.ejemplares_totales),
                               str(self.ejemplares_disponibles)])

    @classmethod
    def desde_linea(cls, linea):
        """Construye un Libro desde una línea del archivo."""
        i, t, a, c, an, tot, disp = linea.strip().split(SEPARADOR)
        return cls(i, t, a, int(c), int(an), int(tot), int(disp))

    def __str__(self):
        return (f"{self.isbn} | {self.titulo} | {self.autor} | cat {self.id_categoria} | "
                f"{self.anio} | {self.ejemplares_disponibles}/{self.ejemplares_totales}")

### Prestamo, Multa y Reserva

In [ ]:
from datetime import date, timedelta


class Prestamo:
    """Préstamo de un libro a un usuario."""

    def __init__(self, id_prestamo, documento, isbn, fecha_prestamo,
                 fecha_limite, fecha_devolucion=None):
        self.id_prestamo = id_prestamo
        self.documento = documento
        self.isbn = isbn
        self.fecha_prestamo = fecha_prestamo
        self.fecha_limite = fecha_limite
        self.fecha_devolucion = fecha_devolucion

    @staticmethod
    def calcular_fecha_limite(fecha_prestamo, tipo_usuario):
        """Fecha límite según el tipo de usuario (7, 15 o 5 días)."""
        return fecha_prestamo + timedelta(days=tipo_usuario.dias_prestamo)

    def esta_activo(self):
        """True si el libro aún no ha sido devuelto."""
        return self.fecha_devolucion is None

    def esta_vencido(self, fecha_referencia):
        """True si está activo y la fecha de referencia supera la fecha límite."""
        return self.esta_activo() and fecha_referencia > self.fecha_limite

    def dias_retraso(self):
        """Días de retraso al devolver (0 si no hubo o no se ha devuelto)."""
        if self.fecha_devolucion is None:
            return 0
        return max(0, (self.fecha_devolucion - self.fecha_limite).days)

    def duracion_real(self):
        """Duración real del préstamo en días (None si sigue activo)."""
        if self.fecha_devolucion is None:
            return None
        return (self.fecha_devolucion - self.fecha_prestamo).days

    def to_linea(self):
        """Serializa el préstamo a una línea delimitada por pipe."""
        dev = self.fecha_devolucion.isoformat() if self.fecha_devolucion else ""
        return SEPARADOR.join([str(self.id_prestamo), self.documento, self.isbn,
                               self.fecha_prestamo.isoformat(),
                               self.fecha_limite.isoformat(), dev])

    @classmethod
    def desde_linea(cls, linea):
        """Construye un Prestamo desde una línea del archivo."""
        i, d, isbn, fp, fl, fd = linea.rstrip("\n").split(SEPARADOR)
        return cls(int(i), d, isbn, date.fromisoformat(fp),
                   date.fromisoformat(fl),
                   date.fromisoformat(fd) if fd else None)

    def __str__(self):
        dev = self.fecha_devolucion or "pendiente"
        return (f"#{self.id_prestamo} | usuario {self.documento} | libro {self.isbn} | "
                f"{self.fecha_prestamo} -> límite {self.fecha_limite} | devuelto: {dev}")


class Multa:
    """Multa generada por devolución tardía."""

    def __init__(self, id_multa, id_prestamo, documento, dias_retraso,
                 valor, pagada=False):
        self.id_multa = id_multa
        self.id_prestamo = id_prestamo
        self.documento = documento
        self.dias_retraso = dias_retraso
        self.valor = valor
        self.pagada = pagada

    @staticmethod
    def calcular_valor(dias_retraso):
        """Valor de la multa: COP 1.000 por día de retraso."""
        return dias_retraso * MULTA_POR_DIA

    def pagar(self):
        """Marca la multa como pagada."""
        self.pagada = True

    def to_linea(self):
        """Serializa la multa a una línea delimitada por pipe."""
        return SEPARADOR.join([str(self.id_multa), str(self.id_prestamo),
                               self.documento, str(self.dias_retraso),
                               str(self.valor), "1" if self.pagada else "0"])

    @classmethod
    def desde_linea(cls, linea):
        """Construye una Multa desde una línea del archivo."""
        i, p, d, dr, v, pg = linea.strip().split(SEPARADOR)
        return cls(int(i), int(p), d, int(dr), int(v), pg == "1")

    def __str__(self):
        estado = "pagada" if self.pagada else "pendiente"
        return (f"Multa #{self.id_multa} | préstamo {self.id_prestamo} | "
                f"{self.dias_retraso} días | ${self.valor:,} COP | {estado}")


class Reserva:
    """Reserva de un libro sin ejemplares disponibles (cola por orden de llegada)."""

    def __init__(self, id_reserva, documento, isbn, fecha,
                 estado=EstadoReserva.ACTIVA):
        self.id_reserva = id_reserva
        self.documento = documento
        self.isbn = isbn
        self.fecha = fecha
        self.estado = estado

    def cancelar(self):
        """Cancela la reserva."""
        self.estado = EstadoReserva.CANCELADA

    def atender(self):
        """Marca la reserva como atendida."""
        self.estado = EstadoReserva.ATENDIDA

    def to_linea(self):
        """Serializa la reserva a una línea delimitada por pipe."""
        return SEPARADOR.join([str(self.id_reserva), self.documento, self.isbn,
                               self.fecha.isoformat(), self.estado.value])

    @classmethod
    def desde_linea(cls, linea):
        """Construye una Reserva desde una línea del archivo."""
        i, d, isbn, f, e = linea.strip().split(SEPARADOR)
        return cls(int(i), d, isbn, date.fromisoformat(f), EstadoReserva(e))

    def __str__(self):
        return (f"Reserva #{self.id_reserva} | usuario {self.documento} | "
                f"libro {self.isbn} | {self.fecha} | {self.estado.value}")

## Sección 2 — Repositorios
Capa de persistencia: leen y escriben archivos `.txt` delimitados por `|`. No contienen reglas de negocio.

### ArchivoUtil y RepositorioBase

In [ ]:
class ArchivoUtil:
    """Utilidades de lectura y escritura de archivos de texto."""

    @staticmethod
    def leer_lineas(ruta):
        """Lee las líneas no vacías de un archivo. Si no existe, retorna lista vacía."""
        if not os.path.exists(ruta):
            return []
        with open(ruta, "r", encoding="utf-8") as f:
            return [l.rstrip("\n") for l in f if l.strip()]

    @staticmethod
    def escribir_lineas(ruta, lineas):
        """Sobrescribe el archivo con las líneas dadas."""
        with open(ruta, "w", encoding="utf-8") as f:
            for l in lineas:
                f.write(l + "\n")

    @staticmethod
    def agregar_linea(ruta, linea):
        """Agrega una línea al final del archivo."""
        with open(ruta, "a", encoding="utf-8") as f:
            f.write(linea + "\n")


class RepositorioBase:
    """Repositorio genérico: CRUD sobre un archivo de texto.

    Las subclases definen la clase del modelo y el nombre del atributo
    que actúa como identificador único.
    """

    def __init__(self, ruta, clase_modelo, atributo_id):
        self.ruta = ruta
        self.clase_modelo = clase_modelo
        self.atributo_id = atributo_id

    def listar(self):
        """Retorna todas las entidades almacenadas."""
        return [self.clase_modelo.desde_linea(l) for l in ArchivoUtil.leer_lineas(self.ruta)]

    def guardar_todos(self, entidades):
        """Reemplaza el contenido del archivo con las entidades dadas."""
        ArchivoUtil.escribir_lineas(self.ruta, [e.to_linea() for e in entidades])

    def buscar_por_id(self, valor):
        """Retorna la entidad cuyo identificador coincide, o None."""
        for e in self.listar():
            if str(getattr(e, self.atributo_id)) == str(valor):
                return e
        return None

    def agregar(self, entidad):
        """Agrega una entidad. Lanza ValueError si el identificador ya existe."""
        if self.buscar_por_id(getattr(entidad, self.atributo_id)) is not None:
            raise ValueError(f"Ya existe un registro con {self.atributo_id} = "
                             f"{getattr(entidad, self.atributo_id)}")
        ArchivoUtil.agregar_linea(self.ruta, entidad.to_linea())

    def actualizar(self, entidad):
        """Reemplaza la entidad con el mismo identificador. Lanza ValueError si no existe."""
        clave = str(getattr(entidad, self.atributo_id))
        entidades = self.listar()
        for i, e in enumerate(entidades):
            if str(getattr(e, self.atributo_id)) == clave:
                entidades[i] = entidad
                self.guardar_todos(entidades)
                return
        raise ValueError(f"No existe un registro con {self.atributo_id} = {clave}")

    def siguiente_id(self):
        """Siguiente identificador numérico (máximo actual + 1)."""
        ids = [int(getattr(e, self.atributo_id)) for e in self.listar()]
        return (max(ids) + 1) if ids else 1

### UsuarioRepository

In [ ]:
class UsuarioRepository(RepositorioBase):
    """Persistencia de usuarios (usuarios.txt)."""

    def __init__(self, ruta=None):
        super().__init__(ruta or RUTA_USUARIOS, Usuario, "documento")

    def buscar_por_documento(self, documento):
        """Busca un usuario por número de documento."""
        return self.buscar_por_id(documento)

### CategoriaRepository

In [ ]:
class CategoriaRepository(RepositorioBase):
    """Persistencia de categorías (categorias.txt)."""

    def __init__(self, ruta=None):
        super().__init__(ruta or RUTA_CATEGORIAS, Categoria, "id_categoria")

    def buscar_por_nombre(self, nombre):
        """Busca una categoría por nombre (sin distinguir mayúsculas)."""
        for c in self.listar():
            if c.nombre.lower() == nombre.strip().lower():
                return c
        return None

### LibroRepository

In [ ]:
class LibroRepository(RepositorioBase):
    """Persistencia de libros (libros.txt)."""

    def __init__(self, ruta=None):
        super().__init__(ruta or RUTA_LIBROS, Libro, "isbn")

    def buscar_por_isbn(self, isbn):
        """Busca un libro por ISBN."""
        return self.buscar_por_id(isbn)

    def buscar_por_titulo(self, texto):
        """Libros cuyo título contiene el texto (sin distinguir mayúsculas)."""
        t = texto.strip().lower()
        return [l for l in self.listar() if t in l.titulo.lower()]

    def buscar_por_categoria(self, id_categoria):
        """Libros de una categoría."""
        return [l for l in self.listar() if l.id_categoria == int(id_categoria)]

    def listar_disponibles(self):
        """Libros con al menos un ejemplar disponible."""
        return [l for l in self.listar() if l.hay_disponibles()]

### PrestamoRepository

In [ ]:
class PrestamoRepository(RepositorioBase):
    """Persistencia de préstamos (prestamos.txt)."""

    def __init__(self, ruta=None):
        super().__init__(ruta or RUTA_PRESTAMOS, Prestamo, "id_prestamo")

    def listar_por_usuario(self, documento):
        """Todos los préstamos de un usuario, en orden cronológico."""
        propios = [p for p in self.listar() if p.documento == documento]
        return sorted(propios, key=lambda p: (p.fecha_prestamo, p.id_prestamo))

    def listar_activos(self):
        """Préstamos aún no devueltos."""
        return [p for p in self.listar() if p.esta_activo()]

    def listar_activos_por_usuario(self, documento):
        """Préstamos activos de un usuario."""
        return [p for p in self.listar_por_usuario(documento) if p.esta_activo()]

    def listar_vencidos(self, fecha_referencia):
        """Préstamos activos vencidos a una fecha dada."""
        return [p for p in self.listar() if p.esta_vencido(fecha_referencia)]

### MultaRepository

In [ ]:
class MultaRepository(RepositorioBase):
    """Persistencia de multas (multas.txt)."""

    def __init__(self, ruta=None):
        super().__init__(ruta or RUTA_MULTAS, Multa, "id_multa")

    def listar_por_usuario(self, documento):
        """Todas las multas de un usuario."""
        return [m for m in self.listar() if m.documento == documento]

    def listar_pendientes_por_usuario(self, documento):
        """Multas sin pagar de un usuario."""
        return [m for m in self.listar_por_usuario(documento) if not m.pagada]

    def buscar_por_prestamo(self, id_prestamo):
        """Multa asociada a un préstamo, o None."""
        for m in self.listar():
            if m.id_prestamo == int(id_prestamo):
                return m
        return None

### ReservaRepository

In [ ]:
class ReservaRepository(RepositorioBase):
    """Persistencia de reservas (reservas.txt)."""

    def __init__(self, ruta=None):
        super().__init__(ruta or RUTA_RESERVAS, Reserva, "id_reserva")

    def listar_activas_por_libro(self, isbn):
        """Reservas activas de un libro, por orden de llegada."""
        activas = [r for r in self.listar()
                   if r.isbn == isbn and r.estado == EstadoReserva.ACTIVA]
        return sorted(activas, key=lambda r: (r.fecha, r.id_reserva))

    def buscar_activa(self, documento, isbn):
        """Reserva activa de un usuario para un libro, o None."""
        for r in self.listar_activas_por_libro(isbn):
            if r.documento == documento:
                return r
        return None

## Prueba rápida de repositorios
Usa una carpeta temporal, no toca los datos de Drive.

In [ ]:
import tempfile

_tmp = tempfile.mkdtemp() + "/"
u_repo = UsuarioRepository(_tmp + "u.txt")
c_repo = CategoriaRepository(_tmp + "c.txt")
l_repo = LibroRepository(_tmp + "l.txt")
p_repo = PrestamoRepository(_tmp + "p.txt")
m_repo = MultaRepository(_tmp + "m.txt")
r_repo = ReservaRepository(_tmp + "r.txt")

hoy = date.today()

# Usuarios
u = Usuario("1001", "Ana Pérez", TipoUsuario.ESTUDIANTE, "ana@mail.com", "3001234567")
u_repo.agregar(u)
try:
    u_repo.agregar(u); raise AssertionError("debió fallar")
except ValueError:
    pass
u.estado = EstadoUsuario.SUSPENDIDO
u_repo.actualizar(u)
assert u_repo.buscar_por_documento("1001").estado == EstadoUsuario.SUSPENDIDO

# Categorías y libros
c_repo.agregar(Categoria(c_repo.siguiente_id(), "Ficción"))
l_repo.agregar(Libro("111", "Cien años de soledad", "García Márquez", 1, 1967, 2))
l_repo.agregar(Libro("222", "El coronel no tiene quien le escriba", "García Márquez", 1, 1961, 1, 0))
assert len(l_repo.buscar_por_titulo("soledad")) == 1
assert len(l_repo.buscar_por_categoria(1)) == 2
assert [l.isbn for l in l_repo.listar_disponibles()] == ["111"]

# Préstamos
p_repo.agregar(Prestamo(p_repo.siguiente_id(), "1001", "111", hoy, hoy + timedelta(days=7)))
p_repo.agregar(Prestamo(p_repo.siguiente_id(), "1001", "222", hoy, hoy + timedelta(days=7), hoy))
assert len(p_repo.listar_por_usuario("1001")) == 2
assert len(p_repo.listar_activos()) == 1
assert len(p_repo.listar_vencidos(hoy + timedelta(days=8))) == 1

# Multas
m_repo.agregar(Multa(m_repo.siguiente_id(), 1, "1001", 3, Multa.calcular_valor(3)))
assert len(m_repo.listar_pendientes_por_usuario("1001")) == 1
m = m_repo.buscar_por_prestamo(1); m.pagar(); m_repo.actualizar(m)
assert len(m_repo.listar_pendientes_por_usuario("1001")) == 0

# Reservas (orden de llegada)
r_repo.agregar(Reserva(1, "1001", "222", hoy))
r_repo.agregar(Reserva(2, "1002", "222", hoy))
assert [r.id_reserva for r in r_repo.listar_activas_por_libro("222")] == [1, 2]

print("✔ Repositorios OK")